source: https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html

In [ ]:
# Core scverse libraries
import scanpy as sc
import anndata as ad

import os

sc.settings.set_figure_params(dpi=50, facecolor="white")

In [ ]:
adata = sc.read_h5ad(os.path.join("..", "datasets", "GSM6592055_M8.h5ad"))
print(adata.obs["Sample_ID"].value_counts())
adata

# Quality Control

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.upper().str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.upper().str.contains("^HB[^(P)]")

In [ ]:
print(f"Mitochondrial genes: {round(adata.var['mt'].sum()/adata.n_vars *100, 1)}%")
print(f"Ribosomal genes: {round(adata.var['ribo'].sum()/adata.n_vars *100, 1)}%")
print(f"Hemoglobin genes: {round(adata.var['hb'].sum()/adata.n_vars *100, 1)}%")

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=3)

# Doublet Detection

In [ ]:
sc.pp.scrublet(adata)

In [ ]:
print(adata.obs["predicted_doublet"].sum())

# Normalization

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()

# Normalizing to 1e4 total counts
sc.pp.normalize_total(adata, target_sum=1e4)
# Logarithmize the data
sc.pp.log1p(adata)

# Feature selection

In [ ]:
adata.n_vars

In [ ]:
sc.pp.highly_variable_genes(adata)
sc.pl.highly_variable_genes(adata)

# Dimensionality Reduction

In [ ]:
sc.tl.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    adata,
    color=["cell_type", "cell_type", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=4,
)

# Nearest neighbor graph construction and visualization

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color="cell_type",
    # Setting a smaller point size to get prevent overlap
    size=4,
)

# Clustering

In [ ]:
import numpy as np
np.random.seed(1)

# Using the igraph implementation and a fixed number of iterations can be significantly faster, especially for larger datasets
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=["leiden"])

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    adata,
    color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"],
    wspace=0.5,
    ncols=2,
)